# Benchmark 2.0 — Part A (Google Colab)
## MobileNetV3Large · MobileNetV2 · EfficientNetB0

**Jalankan notebook ini di Google Colab dengan T4 GPU.**
Setelah selesai, download `benchmark2_outputs.zip` lalu upload ke notebook Merge.

| Model | Catatan |
|---|---|
| MobileNetV3Large | Baseline SapiKenal production |
| MobileNetV2 | Paling ringan |
| EfficientNetB0 | Compound scaling |

⏱ Estimasi: **~60–90 menit** di T4

In [ ]:
# ========================================
# Setup Lingkungan & Verifikasi GPU
# ========================================
!pip install -q roboflow

import os, gc, time, math, warnings, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from pathlib import Path
from collections import Counter

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
)
from scipy import stats as scipy_stats
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical, load_img, img_to_array

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 150
sns.set_style("whitegrid")

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"GPU: {gpus[0].name}")
else:
    print("GPU tidak terdeteksi — aktifkan T4/P100 di Runtime settings.")
print(f"TensorFlow: {tf.__version__}")

In [ ]:
# ========================================
# Download & Gabung Seluruh Split Dataset
# ========================================
from roboflow import Roboflow

ROBOFLOW_API_KEY = "oSqbdWgbnJkiVfOWl18o"
WORKSPACE_NAME   = "valll-rtclj"
PROJECT_NAME     = "cattle-breeds-zrtzt"
DATASET_VERSION  = 2

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
dataset = rf.workspace(WORKSPACE_NAME).project(PROJECT_NAME).version(DATASET_VERSION).download("folder")
DATASET_DIR = dataset.location

IMAGE_SIZE = (224, 224)

all_paths, all_labels = [], []
for split in ["train", "valid", "test"]:
    split_dir = os.path.join(DATASET_DIR, split)
    if not os.path.exists(split_dir):
        continue
    for cls in sorted(os.listdir(split_dir)):
        cls_dir = os.path.join(split_dir, cls)
        if not os.path.isdir(cls_dir):
            continue
        for img in sorted(os.listdir(cls_dir)):
            if img.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp")):
                all_paths.append(os.path.join(cls_dir, img))
                all_labels.append(cls)

print(f"Total gambar: {len(all_paths)}")
class_names = sorted(set(all_labels))
NUM_CLASSES  = len(class_names)
label_counts = Counter(all_labels)
print(f"Kelas ({NUM_CLASSES}): {class_names}")
for c in class_names:
    print(f"  {c}: {label_counts[c]}")

print("\nMemuat gambar ke memori...")
X_all = np.zeros((len(all_paths), *IMAGE_SIZE, 3), dtype=np.float32)
for i, p in enumerate(all_paths):
    X_all[i] = img_to_array(load_img(p, target_size=IMAGE_SIZE))

label_to_idx = {n: i for i, n in enumerate(class_names)}
y_all     = np.array([label_to_idx[l] for l in all_labels])
y_all_cat = to_categorical(y_all, NUM_CLASSES)
print(f"X={X_all.shape}, y={y_all.shape}")

In [ ]:
# ========================================
# Konfigurasi & Fungsi Utilitas
# ========================================
BATCH_SIZE         = 16
PHASE1_EPOCHS      = 30
PHASE2_EPOCHS      = 20
PHASE1_LR          = 1e-3
PHASE2_LR          = 1e-5
N_FOLDS            = 5
UNFREEZE_FRACTION  = 0.3
RANDOM_STATE       = 42

OUTPUT_DIR = Path("/content/benchmark2_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def build_model(base_fn, preprocess_fn, num_classes, input_shape=(224, 224, 3), name="model"):
    base = base_fn(weights="imagenet", include_top=False, input_shape=input_shape)
    base.trainable = False
    x   = GlobalAveragePooling2D(name="gap")(base.output)
    x   = Dense(256, activation="relu", name="dense_head")(x)
    x   = Dropout(0.3, name="dropout_head")(x)
    out = Dense(num_classes, activation="softmax", name="output")(x)
    m   = Model(inputs=base.input, outputs=out, name=name)
    m.compile(optimizer=tf.keras.optimizers.Adam(PHASE1_LR),
              loss="categorical_crossentropy", metrics=["accuracy"])
    return m


def unfreeze_top(model, fraction=0.3):
    base_layers = []
    for layer in model.layers:
        if layer.name == "gap":
            break
        base_layers.append(layer)
    for layer in base_layers:
        layer.trainable = False
    n = max(1, int(len(base_layers) * fraction))
    for layer in base_layers[-n:]:
        if not isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = True
    for layer in model.layers:
        if layer.name in ("dense_head", "dropout_head", "output"):
            layer.trainable = True
    model.compile(optimizer=tf.keras.optimizers.Adam(PHASE2_LR),
                  loss="categorical_crossentropy", metrics=["accuracy"])


def get_callbacks():
    return [
        EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=3, min_lr=1e-7, verbose=0),
    ]


def eval_model(model, X_val, y_val, preprocess_fn):
    X_p   = preprocess_fn(X_val.copy())
    probs = model.predict(X_p, verbose=0)
    yp    = np.argmax(probs, axis=1)
    return {
        "accuracy" : accuracy_score(y_val, yp),
        "precision": precision_score(y_val, yp, average="macro", zero_division=0),
        "recall"   : recall_score(y_val, yp, average="macro", zero_division=0),
        "f1"       : f1_score(y_val, yp, average="macro", zero_division=0),
        "y_pred"   : yp,
        "pc_p"     : precision_score(y_val, yp, average=None, zero_division=0),
        "pc_r"     : recall_score(y_val, yp, average=None, zero_division=0),
        "pc_f1"    : f1_score(y_val, yp, average=None, zero_division=0),
    }

print("Helpers siap.")

## Training: 5-Fold CV × 3 Model × 2 Phase

> Jika sesi Colab terputus, set `RESUME = True` di cell berikut lalu jalankan ulang.

In [ ]:
# ========================================
# Training: 5-Fold CV × 3 Model (Part A) × 2 Phase
# ========================================
# Set RESUME = True jika ingin lanjutkan dari checkpoint yang sudah ada
RESUME = False

MODEL_CONFIGS = {
    "MobileNetV3Large": {
        "base_fn"     : tf.keras.applications.MobileNetV3Large,
        "preprocess_fn": tf.keras.applications.mobilenet_v3.preprocess_input,
    },
    "MobileNetV2": {
        "base_fn"     : tf.keras.applications.MobileNetV2,
        "preprocess_fn": tf.keras.applications.mobilenet_v2.preprocess_input,
    },
    "EfficientNetB0": {
        "base_fn"     : tf.keras.applications.EfficientNetB0,
        "preprocess_fn": tf.keras.applications.efficientnet.preprocess_input,
    },
}
print(f"Models: {list(MODEL_CONFIGS.keys())}")

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
ckpt = OUTPUT_DIR / "checkpoint_results.pkl"

if RESUME and ckpt.exists():
    with open(ckpt, "rb") as f:
        all_results = pickle.load(f)
    completed = set(all_results.keys())
    print(f"Resume — sudah selesai: {completed}")
else:
    all_results = {}
    completed   = set()

for model_name, cfg in MODEL_CONFIGS.items():
    if model_name in completed:
        print(f"SKIP {model_name} (sudah selesai)")
        continue

    print(f"\n{'='*70}")
    print(f"  MODEL: {model_name}")
    print(f"{'='*70}")

    mr = {
        "phase1": {"acc": [], "prec": [], "rec": [], "f1": [],
                   "y_trues": [], "y_preds": [],
                   "pc_p": [], "pc_r": [], "pc_f1": []},
        "phase2": {"acc": [], "prec": [], "rec": [], "f1": [],
                   "y_trues": [], "y_preds": [],
                   "pc_p": [], "pc_r": [], "pc_f1": []},
        "times": [], "class_names": class_names,
    }

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_all, y_all)):
        print(f"  Fold {fold+1}/{N_FOLDS}", end=" ")

        X_tr, X_va = X_all[tr_idx].copy(), X_all[va_idx].copy()
        y_tr_c     = y_all_cat[tr_idx]
        y_va_c     = y_all_cat[va_idx]
        y_va_l     = y_all[va_idx]

        aug = ImageDataGenerator(
            preprocessing_function=cfg["preprocess_fn"],
            rotation_range=25, width_shift_range=0.1, height_shift_range=0.1,
            shear_range=0.1, zoom_range=0.2, horizontal_flip=True,
            brightness_range=[0.8, 1.2], fill_mode="nearest",
        )
        val_gen = ImageDataGenerator(preprocessing_function=cfg["preprocess_fn"])
        steps_tr = math.ceil(len(X_tr) / BATCH_SIZE)
        steps_va = math.ceil(len(X_va) / BATCH_SIZE)

        model = build_model(cfg["base_fn"], cfg["preprocess_fn"],
                            NUM_CLASSES, name=model_name)
        t0 = time.time()

        # Phase 1 — Feature Extraction
        model.fit(
            aug.flow(X_tr, y_tr_c, batch_size=BATCH_SIZE, seed=RANDOM_STATE),
            steps_per_epoch=steps_tr, epochs=PHASE1_EPOCHS,
            validation_data=val_gen.flow(X_va, y_va_c, batch_size=BATCH_SIZE, shuffle=False),
            validation_steps=steps_va, callbacks=get_callbacks(), verbose=0,
        )
        m1 = eval_model(model, X_va, y_va_l, cfg["preprocess_fn"])
        for k in ["acc", "prec", "rec", "f1"]:
            mr["phase1"][k].append(m1[{"acc":"accuracy","prec":"precision","rec":"recall","f1":"f1"}[k]])
        mr["phase1"]["y_trues"].append(y_va_l)
        mr["phase1"]["y_preds"].append(m1["y_pred"])
        mr["phase1"]["pc_p"].append(m1["pc_p"])
        mr["phase1"]["pc_r"].append(m1["pc_r"])
        mr["phase1"]["pc_f1"].append(m1["pc_f1"])
        print(f"| P1 acc={m1['accuracy']:.3f} f1={m1['f1']:.3f}", end=" ")

        # Phase 2 — Fine-Tuning
        unfreeze_top(model, UNFREEZE_FRACTION)
        model.fit(
            aug.flow(X_tr, y_tr_c, batch_size=BATCH_SIZE, seed=RANDOM_STATE + fold),
            steps_per_epoch=steps_tr, epochs=PHASE2_EPOCHS,
            validation_data=val_gen.flow(X_va, y_va_c, batch_size=BATCH_SIZE, shuffle=False),
            validation_steps=steps_va, callbacks=get_callbacks(), verbose=0,
        )
        elapsed = time.time() - t0
        mr["times"].append(elapsed)
        m2 = eval_model(model, X_va, y_va_l, cfg["preprocess_fn"])
        for k in ["acc", "prec", "rec", "f1"]:
            mr["phase2"][k].append(m2[{"acc":"accuracy","prec":"precision","rec":"recall","f1":"f1"}[k]])
        mr["phase2"]["y_trues"].append(y_va_l)
        mr["phase2"]["y_preds"].append(m2["y_pred"])
        mr["phase2"]["pc_p"].append(m2["pc_p"])
        mr["phase2"]["pc_r"].append(m2["pc_r"])
        mr["phase2"]["pc_f1"].append(m2["pc_f1"])
        print(f"| P2 acc={m2['accuracy']:.3f} f1={m2['f1']:.3f} | {elapsed:.0f}s")

        if m2["accuracy"] >= max(mr["phase2"]["acc"]):
            model.save(OUTPUT_DIR / f"best_{model_name}.keras")

        del model, X_tr, X_va
        tf.keras.backend.clear_session()
        gc.collect()

    all_results[model_name] = mr
    p2 = mr["phase2"]["acc"]
    print(f"  >> {model_name} P2: {np.mean(p2)*100:.2f}% +/- {np.std(p2)*100:.2f}%")
    with open(ckpt, "wb") as f:
        pickle.dump(all_results, f)

print("\nSEMUA MODEL SELESAI!")

# Simpan hasil ke CSV
rows = []
for name, r in all_results.items():
    for ph, pk in [("Feature Extraction", "phase1"), ("Fine-Tuning", "phase2")]:
        p = r[pk]
        rows.append({
            "Model": name, "Phase": ph,
            "Acc_mean": np.mean(p["acc"]), "Acc_std": np.std(p["acc"]),
            "F1_mean" : np.mean(p["f1"]),  "F1_std" : np.std(p["f1"]),
            "Prec_mean": np.mean(p["prec"]),"Prec_std": np.std(p["prec"]),
            "Rec_mean" : np.mean(p["rec"]), "Rec_std" : np.std(p["rec"]),
        })
pd.DataFrame(rows).to_csv(OUTPUT_DIR / "results_partial.csv", index=False)
print("Hasil disimpan ke results_partial.csv")

# Simpan pickle lengkap untuk merge
with open(OUTPUT_DIR / "all_results.pkl", "wb") as f:
    pickle.dump(all_results, f)
print("Pickle disimpan ke all_results.pkl — diperlukan oleh notebook Merge.")

In [ ]:
# ========================================
# Evaluasi Deployment (TFLite)
# ========================================
deployment = {}

for name in MODEL_CONFIGS:
    bp = OUTPUT_DIR / f"best_{name}.keras"
    if not bp.exists():
        print(f"  {name}: model tidak ditemukan, skip")
        continue

    print(f"  Converting {name}...", end=" ")
    model = tf.keras.models.load_model(bp)

    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    tflite_bytes = converter.convert()
    tflite_path = OUTPUT_DIR / f"{name}.tflite"
    with open(tflite_path, "wb") as f:
        f.write(tflite_bytes)

    tflite_mb = os.path.getsize(tflite_path) / 1024**2
    keras_mb  = os.path.getsize(bp) / 1024**2
    params    = model.count_params()

    interp = tf.lite.Interpreter(model_path=str(tflite_path))
    interp.allocate_tensors()
    inp = interp.get_input_details()
    dummy = np.random.rand(1, 224, 224, 3).astype(np.float32)
    lats = []
    for _ in range(110):
        t0 = time.time()
        interp.set_tensor(inp[0]["index"], dummy)
        interp.invoke()
        lats.append((time.time() - t0) * 1000)
    avg_lat = np.mean(lats[10:])

    deployment[name] = {
        "params_m"  : params / 1e6,
        "keras_mb"  : keras_mb,
        "tflite_mb" : tflite_mb,
        "latency_ms": avg_lat,
    }
    print(f"{params/1e6:.2f}M | TFLite {tflite_mb:.1f}MB | {avg_lat:.1f}ms")

    del model, interp
    tf.keras.backend.clear_session()
    gc.collect()

# Simpan ke pickle untuk merge
with open(OUTPUT_DIR / "deployment.pkl", "wb") as f:
    pickle.dump(deployment, f)

for name, d in deployment.items():
    print(f"  {name}: params={d['params_m']:.2f}M | TFLite={d['tflite_mb']:.1f}MB | lat={d['latency_ms']:.1f}ms")

In [ ]:
# ========================================
# Download Hasil sebagai ZIP
# ========================================
import shutil
zip_path = shutil.make_archive("/content/benchmark2_outputs", "zip", "/content", "benchmark2_outputs")
print(f"ZIP: {zip_path}")
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("File tersedia di:", zip_path)